# Complete PER Pipeline Tutorial

Dieses Tutorial zeigt den vollständigen Workflow des Pauli-Lindblad PER Frameworks:

1. **VQE Circuit-Erstellung** (bereits vorbereitet)
2. **Resource Planning** - Circuit-Zählung vor der Ausführung
3. **Subgraph-Verwendung** - Nur eine Teilmenge der Qubits nutzen
4. **Sparse Pauli Tomographie** - Noise-Charakterisierung
5. **PER Execution** - Fehlerminderung mit gelernten Modellen
6. **Results Analysis** - Vergleich mit idealen Werten

Backend: **FakeWashingtonV2** (127 Qubits, davon nutzen wir 4)

## 1. Setup und VQE Circuit Vorbereitung

In [ ]:
import sys; sys.path.append('../..')
import numpy as np
import matplotlib.pyplot as plt

# Qiskit imports
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime.fake_provider import FakeWashingtonV2
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.quantum_info import SparsePauliOp
from scipy.linalg import eigh

# PER Framework imports
sys.path.append('../../pauli_lindblad_per')
from tomography.experiment import SparsePauliTomographyExperiment
from src.calculate_runs import count_tomography_runs_from_repo, count_per_runs

# VQE Helper Functions
from src.VQE_functions import optimize_energy, MFIM_Hamiltonian, build_hva_layers
from src.plot_helpers import plot_topology

print("✓ All imports successful!")

### 1.1 Topology und Hamiltonian definieren

In [ ]:
# 4-Qubit Square Topology
topology = np.array([[0, 0], [1, 0], [1, 1], [0, 1]])
ax = plot_topology(topology)
plt.title("4-Qubit Square Topology")
plt.show()

# Hamiltonian Parameters
hx = 1.0
hz = 1.0  
J = -1.0

Hamiltonian = MFIM_Hamiltonian(J=J, hx=hx, hz=hz, topology=topology)
exact_energy = np.min(eigh(Hamiltonian.to_matrix(), eigvals_only=True))

print(f"\nHamiltonian: Mixed-Field Ising Model")
print(f"Parameters: hx={hx}, hz={hz}, J={J}")
print(f"Exact ground state energy: {exact_energy:.6f}")

### 1.2 Backend Setup - FakeWashingtonV2 mit Subgraph

**Wichtig**: FakeWashingtonV2 hat 127 Qubits, aber wir verwenden nur **4 Qubits** für unser Experiment.

Das Framework erkennt automatisch den Subgraph-Modus!

In [ ]:
# FakeWashingtonV2 Backend laden (127 Qubits)
fake_backend = FakeWashingtonV2()

# Noise Model extrahieren
noise_model = NoiseModel.from_backend(fake_backend)

# Simulator mit Noise Model
backend = AerSimulator(noise_model=noise_model)

# Wähle 4 zusammenhängende Qubits aus FakeWashington
# Basierend auf der Coupling Map - wir nehmen Qubits mit guter Konnektivität
used_qubits = [0, 1, 14, 18]  # Bildet einen zusammenhängenden Subgraph
inst_map = used_qubits  # 1:1 Mapping für dieses Beispiel

print(f"Backend: {fake_backend.name}")
print(f"Total qubits on backend: {fake_backend.num_qubits}")
print(f"Used qubits (subgraph): {used_qubits}")
print(f"\n✓ Subgraph-Modus wird automatisch erkannt!")

# Visualisiere relevante Backend-Eigenschaften
coupling_map = fake_backend.coupling_map
print(f"\nRelevant connections for our qubits:")
for edge in coupling_map.get_edges():
    if edge[0] in used_qubits and edge[1] in used_qubits:
        print(f"  {edge[0]} ↔ {edge[1]}")

### 1.3 VQE Optimierung (vom vorherigen Notebook)

In [ ]:
num_layers = 1
initial_params = [1, 1, 1]
bounds = [(-np.pi, np.pi)] * 3 * num_layers

# HVA Circuit erstellen
hva_layers, params = build_hva_layers(topology=topology, num_layers=num_layers)

# Noise-freier Simulator für Optimierung
ideal_backend = AerSimulator()

print("Starting VQE optimization...")
opt_params, energy, HelperInfo = optimize_energy(
    initial_params=initial_params,
    Hamiltonian=Hamiltonian,
    topology=topology,
    num_layers=num_layers,
    backend=ideal_backend,
    mode="measurement",
    bounds=bounds
)

# Optimierten Circuit erstellen
values = {p: v for p, v in zip(params, opt_params)}
qc_optimized = hva_layers.assign_parameters(values)

print("\n" + "="*80)
print("VQE OPTIMIZATION RESULTS")
print("="*80)
print(f"Exact ground state energy:     {exact_energy:.8f}")
print(f"VQE optimized energy:          {energy:.8f}")
print(f"Relative error:                {np.abs(energy - exact_energy) / np.abs(exact_energy) * 100:.4f}%")
print(f"\nInitial parameters: {initial_params}")
print(f"Optimized parameters: {opt_params}")
print("="*80)

print("\n" + qc_optimized.draw(output='text', fold=100))

## 2. Resource Planning - Circuit-Zählung im Vorhinein

**Wichtig**: Vor der Ausführung auf echter Hardware sollten wir wissen, wie viele Circuits generiert werden!

Nutze `calculate_runs.py` um die benötigten Ressourcen zu planen.

In [ ]:
# Parameter für Tomografie
tomo_samples = 1000
tomo_single_samples = 2000
tomo_depths = [1, 2, 4, 8]
shots_per_circuit = 1024

# Zähle Tomografie Circuits
tomo_stats = count_tomography_runs_from_repo(
    qc=qc_optimized,
    inst_map=inst_map,
    backend=fake_backend,
    used_qubits=used_qubits,
    depths=tomo_depths,
    samples=tomo_samples,
    single_samples=tomo_single_samples,
    shots=shots_per_circuit
)

print("="*80)
print("TOMOGRAPHY RESOURCE ESTIMATION")
print("="*80)
print(f"Total circuits to generate:    {tomo_stats['circuits']:,}")
print(f"Total backend runs:            {tomo_stats['backend_runs']:,}")
print(f"Estimated time (1 run/sec):    {tomo_stats['backend_runs']/3600:.2f} hours")
print(f"\nDetails:")
for key, value in tomo_stats['details'].items():
    print(f"  {key:25}: {value}")
print("="*80)

In [ ]:
# Parameter für PER
per_samples = 3000
per_noise_strengths = [0.1, 0.2, 0.3, 0.4]

# Wichtige Observable aus dem Hamiltonian
hamiltonian_paulis = [str(p) for p in Hamiltonian.paulis[:10]]  # Top 10 Terms
diagnostic_paulis = ['ZIII', 'IZII', 'IIZI', 'IIIZ']  # Single-qubit diagnostics
per_observables = hamiltonian_paulis + diagnostic_paulis

print(f"\nPER Observables ({len(per_observables)} total):")
print(f"  From Hamiltonian: {len(hamiltonian_paulis)}")
print(f"  Diagnostic:       {len(diagnostic_paulis)}")

# Zähle PER Circuits
per_stats = count_per_runs(
    qc=qc_optimized,
    pauli_list=per_observables,
    noise_strengths=per_noise_strengths,
    samples=per_samples,
    shots=shots_per_circuit
)

print("\n" + "="*80)
print("PER RESOURCE ESTIMATION")
print("="*80)
print(f"Total circuits to generate:    {per_stats['circuits']:,}")
print(f"Total backend runs:            {per_stats['backend_runs']:,}")
print(f"Estimated time (1 run/sec):    {per_stats['backend_runs']/3600:.2f} hours")
print(f"\nDetails:")
for key, value in per_stats['details'].items():
    print(f"  {key:25}: {value}")
print("="*80)

# Gesamtübersicht
total_circuits = tomo_stats['circuits'] + per_stats['circuits']
total_runs = tomo_stats['backend_runs'] + per_stats['backend_runs']

print("\n" + "="*80)
print("TOTAL RESOURCE REQUIREMENTS")
print("="*80)
print(f"Total circuits:                {total_circuits:,}")
print(f"Total backend runs:            {total_runs:,}")
print(f"Estimated total time:          {total_runs/3600:.2f} hours")
print(f"\n⚠️  Adjust samples if resource budget is limited!")
print("="*80)

## 3. Sparse Pauli Tomographie

Jetzt führen wir die Noise-Charakterisierung durch. Das Framework erkennt automatisch:
- Subgraph-Modus (4 von 127 Qubits)
- Einzigartige Circuit-Layer
- Automatisches Speichern der Ergebnisse

In [ ]:
# Tomographie-Experiment erstellen
circuits_to_characterize = [qc_optimized]  # Kann auch mehrere Circuits sein

experiment = SparsePauliTomographyExperiment(
    circuits=circuits_to_characterize,
    inst_map=inst_map,
    backend=fake_backend,
    used_qubits=used_qubits  # Automatische Subgraph-Erkennung!
)

print("✓ Tomography experiment created")
print(f"  Subgraph mode: {experiment.subgraph}")
print(f"  Used qubits: {used_qubits}")
print(f"  Unique circuit layers: {len(experiment._profiles)}")

In [ ]:
# Benchmarking-Prozeduren generieren
# Für schnelle Tests: weniger Samples
experiment.generate(
    samples=100,        # Reduziert für Tutorial
    single_samples=200,  # Reduziert für Tutorial  
    depths=[1, 2, 4]     # Weniger Tiefen für Tutorial
)

print("✓ Benchmarking procedures generated")
print(f"  Ready to run on backend")

In [ ]:
# Executor Function für Simulation
def simulator_executor(circuits):
    """Simuliert die Circuits mit dem Noise Model."""
    from qiskit import transpile
    
    print(f"Executing {len(circuits)} circuits on simulator...")
    
    # Transpile für Backend
    transpiled = transpile(
        circuits,
        backend=backend,
        initial_layout=used_qubits,
        optimization_level=1
    )
    
    # Ausführen
    results = []
    for circuit in transpiled:
        result = backend.run(circuit, shots=1024).result()
        counts = result.get_counts()
        results.append(counts)
    
    print(f"✓ Execution complete")
    return results

print("✓ Executor function defined")

In [ ]:
# Tomographie ausführen mit automatischem Speichern
print("Starting tomography execution...")
print("Results will be auto-saved to SaveFiles/\n")

experiment.run(
    executor=simulator_executor,
    auto_save=True,
    save_filename="SaveFiles/tutorial_tomography_results.json"
)

print("\n✓ Tomography execution complete!")
print("✓ Results saved to SaveFiles/tutorial_tomography_results.json")

In [ ]:
# Noise-Modelle analysieren und lernen
print("Analyzing noise models...\n")

noise_data_frame = experiment.analyze()

print("="*80)
print("NOISE MODEL ANALYSIS COMPLETE")
print("="*80)
print(f"Learned noise models for {len(experiment._layers)} unique layers")
print(f"\nNoise data frame ready for PER!")
print("="*80)

### 3.1 Noise-Model Visualisierung

In [ ]:
# Visualisiere gelernte Noise-Modelle
for layer_idx, layer_data in enumerate(noise_data_frame.layers):
    print(f"\n{'='*80}")
    print(f"LAYER {layer_idx} ANALYSIS")
    print(f"{'='*80}")
    
    # Model-Koeffizienten
    print("\nModel Coefficients (Noise Generator):")
    layer_data.plot_coeffs(plot_style=2)
    plt.show()
    
    # Infidelitäten
    print("\nInfidelities by Qubit/Pair:")
    layer_data.plot_infidelitites(plot_style=2)
    plt.show()
    
    # Top Error Terms
    coeffs_dict = dict(layer_data.noisemodel.coeffs)
    sorted_terms = sorted(coeffs_dict.items(), key=lambda x: x[1], reverse=True)
    print("\nTop 5 Error Terms:")
    for i, (term, coeff) in enumerate(sorted_terms[:5], 1):
        print(f"  {i}. {term.to_label():10s}: {coeff:.6f}")

print("\n✓ Noise characterization complete!")

## 4. Pauli Error Reconstruction (PER)

Jetzt nutzen wir die gelernten Noise-Modelle für Fehlerminderung!

In [ ]:
# PER-Experiment erstellen mit gelernten Noise-Modellen
test_circuits = [qc_optimized]  # Kann auch andere Circuits sein

per_experiment = experiment.create_per_experiment(test_circuits)

print("✓ PER experiment created with learned noise models")
print(f"  Testing {len(test_circuits)} circuit(s)")

In [ ]:
# Observable Selection
# Wir messen die wichtigsten Hamiltonian-Terme
important_observables = [str(p) for p in Hamiltonian.paulis[:5]]  # Top 5 Terms

# Plus diagnostische Observables
diagnostic_observables = [
    'ZIII', 'IZII', 'IIZI', 'IIIZ',  # Single Z
    'ZZII', 'IZZI', 'IIZZ'            # Neighboring ZZ
]

all_per_observables = important_observables + diagnostic_observables

print(f"PER Observables ({len(all_per_observables)} total):")
print(f"  Hamiltonian terms: {important_observables}")
print(f"  Diagnostic terms:  {diagnostic_observables}")

# Noise-Skalierungsfaktoren
noise_strengths = [0.1, 0.2, 0.3]  # Reduziert für Tutorial

# PER-Circuits generieren  
per_experiment.generate(
    expectations=all_per_observables,
    samples=500,  # Reduziert für Tutorial
    noise_strengths=noise_strengths
)

print(f"\n✓ PER circuits generated")
print(f"  Noise strengths: {noise_strengths}")
print(f"  Samples per strength: 500")

In [ ]:
# PER ausführen mit automatischem Speichern
print("Starting PER execution...")
print("Results will be auto-saved to SaveFiles/\n")

per_experiment.run(
    executor=simulator_executor,
    auto_save=True,
    save_filename="SaveFiles/tutorial_per_results.json"
)

print("\n✓ PER execution complete!")
print("✓ Results saved to SaveFiles/tutorial_per_results.json")

In [ ]:
# PER-Ergebnisse analysieren
print("Analyzing PER results...\n")

per_results = per_experiment.analyze()

print("✓ PER analysis complete!")

## 5. Results Analysis und Validation

In [ ]:
# Detaillierte Auswertung pro Circuit
from qiskit.quantum_info import Statevector

print("="*80)
print("PER RESULTS ANALYSIS")
print("="*80)

for circuit_idx, per_run in enumerate(per_results):
    print(f"\nCircuit {circuit_idx}:")
    print("-" * 80)
    
    # Ideale Erwartungswerte berechnen
    psi_ideal = Statevector.from_instruction(test_circuits[circuit_idx])
    ideal_energy = np.real(psi_ideal.expectation_value(Hamiltonian))
    
    # PER-rekonstruierte Energie
    per_energy = 0
    per_data_summary = []
    
    for pauli_str, coeff in zip([str(p) for p in Hamiltonian.paulis], Hamiltonian.coeffs):
        if pauli_str in per_run.data:
            per_data = per_run.data[pauli_str]
            
            # Exponentieller Fit
            ideal_expectation, decay_rate = per_data.fit()
            per_energy += coeff * ideal_expectation
            
            # Für Top-5 Hamiltonian-Terms speichern
            if pauli_str in important_observables:
                ideal_val = np.real(psi_ideal.expectation_value(
                    SparsePauliOp.from_list([(pauli_str, 1.0)])
                ))
                per_data_summary.append((
                    pauli_str,
                    ideal_val,
                    ideal_expectation,
                    decay_rate
                ))
    
    # Energie-Vergleich
    print(f"\nEnergy Reconstruction:")
    print(f"  Exact (noise-free):     {exact_energy:.8f}")
    print(f"  VQE ideal:              {ideal_energy:.8f}")
    print(f"  PER reconstructed:      {per_energy:.8f}")
    print(f"  \nPER vs Exact error:     {abs(per_energy - exact_energy):.8f}")
    print(f"  PER vs Exact (%):       {abs(per_energy - exact_energy) / abs(exact_energy) * 100:.4f}%")
    
    # Observable-Vergleich
    print(f"\nTop Observable Reconstructions:")
    print(f"  {'Observable':<12} {'Ideal':<12} {'PER':<12} {'Error':<12} {'Decay Rate':<12}")
    print("  " + "-"*70)
    for obs, ideal, per_val, decay in per_data_summary:
        error = abs(per_val - ideal)
        print(f"  {obs:<12} {ideal:>11.6f} {per_val:>11.6f} {error:>11.6f} {decay:>11.6f}")

print("\n" + "="*80)

In [ ]:
# PER Noise-Skalierung visualisieren
print("\nPER Noise Scaling Analysis:\n")

for circuit_idx, per_run in enumerate(per_results):
    # Plotte die wichtigsten Observables
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, observable in enumerate(important_observables[:4]):
        if observable in per_run.data:
            per_data = per_run.data[observable]
            
            # Plot erstellen
            ax = axes[idx]
            ax.plot(
                per_data.get_strengths(),
                per_data.get_expectations(),
                'x',
                markersize=10,
                label='Measured',
                color='blue'
            )
            
            # Exponentieller Fit
            a, b = per_data.fit()
            xlin = np.linspace(0, max(per_data.get_strengths()), 100)
            ax.plot(
                xlin,
                [a * np.exp(b * x) for x in xlin],
                '--',
                label=f'Fit: {a:.3f}·exp({b:.3f}·λ)',
                color='red'
            )
            
            ax.axhline(y=a, color='green', linestyle=':', alpha=0.5, label=f'PER: {a:.3f}')
            ax.set_xlabel('Noise Strength (λ)')
            ax.set_ylabel('Expectation Value')
            ax.set_title(f'Observable: {observable}')
            ax.legend()
            ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'PER Noise Scaling Analysis - Circuit {circuit_idx}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("✓ All visualizations complete!")

## 6. Summary und Best Practices

### Was haben wir gelernt?

1. **Subgraph-Verwendung**: FakeWashingtonV2 hat 127 Qubits, wir nutzen nur 4
   - Automatische Erkennung durch Framework
   - Effiziente Ressourcen-Nutzung

2. **Resource Planning**: Circuit-Zählung vor Ausführung
   - `count_tomography_runs_from_repo()` für Tomografie
   - `count_per_runs()` für PER
   - Ermöglicht Budget-Planung

3. **Automatisches Speichern**: Crash-Recovery
   - Tomografie → `SaveFiles/tutorial_tomography_results.json`
   - PER → `SaveFiles/tutorial_per_results.json`
   - JSON-Format für Lesbarkeit

4. **Kompletter Workflow**:
   - VQE Circuit-Optimierung
   - Noise-Charakterisierung (Tomografie)
   - Fehlerminderung (PER)
   - Validation mit idealen Werten

### Best Practices

```python
# Immer Resource Planning durchführen
stats = count_tomography_runs_from_repo(...)
if stats['backend_runs'] > budget:
    # Reduziere samples/depths

# Nutze Auto-Save
experiment.run(executor, auto_save=True)

# Bei Crash: Einfach neu laden
results = experiment.analyze_from_file()  # Lädt neueste automatisch

# Für echte Hardware: Batch-Ausführung
def robust_executor(circuits):
    # Mit Retry-Logic und Error-Handling
    ...
```

### Next Steps

- **Mehr Circuits**: Teste mit verschiedenen Parametrisierungen
- **Mehr Observables**: Analysiere weitere Hamiltonian-Terme
- **Echte Hardware**: Verwende echte IBM Quantum Backends
- **Optimierung**: Fine-tune Noise-Strengths für bessere Fits


In [ ]:
# Final Check: Zeige was gespeichert wurde
import os
import glob

print("="*80)
print("SAVED FILES IN SaveFiles/")
print("="*80)

saved_files = glob.glob("../../SaveFiles/*.json")
for file in saved_files:
    size = os.path.getsize(file) / 1024  # KB
    print(f"  {os.path.basename(file):50s} | {size:8.1f} KB")

print("\n✓ Tutorial complete!")
print("\nAll results are saved and can be re-analyzed at any time:")
print("  experiment.analyze_from_file('SaveFiles/tutorial_tomography_results.json')")
print("  per_experiment.analyze_from_file('SaveFiles/tutorial_per_results.json')")